In [1]:
import os
from pyhive import hive
import warnings
import pandas as pd
warnings.simplefilter(action='ignore', category=UserWarning)

default_db = 'com490'
hive_server = os.environ.get('HIVE_SERVER','iccluster080.iccluster.epfl.ch:10000')
hadoop_fs = os.environ.get('HADOOP_DEFAULT_FS','hdfs://iccluster067.iccluster.epfl.ch:8020')
username  = os.environ.get('USER', 'anonym')
(hive_host, hive_port) = hive_server.split(':')

conn = hive.connect(
    host=hive_host,
    port=hive_port,
    username=username
)

# create cursor
cur = conn.cursor()

print(f"hadoop hdfs URL is {hadoop_fs}")
print(f"your username is {username}")
print(f"you are connected to {hive_host}:{hive_port}")

hadoop hdfs URL is hdfs://iccluster067.iccluster.epfl.ch:8020
your username is wmaj
you are connected to iccluster080.iccluster.epfl.ch:10000


In [2]:
OBJECT_ID = 1 # Choose the region of interest


"""
1 : Lausanne Region
1696 : Fribourg region
...

"""

'\n1 : Lausanne Region\n1696 : Fribourg region\n...\n\n'

In [3]:
# Enable ESRI UDF
cur.execute(f"""
ADD JARS
    {hadoop_fs}/data/jars/esri-geometry-api-2.2.4.jar
    {hadoop_fs}/data/jars/spatial-sdk-hive-2.2.0.jar
    {hadoop_fs}/data/jars/spatial-sdk-json-2.2.0.jar
""")
cur.execute("CREATE TEMPORARY FUNCTION ST_Point AS 'com.esri.hadoop.hive.ST_Point'")
cur.execute("CREATE TEMPORARY FUNCTION ST_Distance AS 'com.esri.hadoop.hive.ST_Distance'")
cur.execute("CREATE TEMPORARY FUNCTION ST_SetSRID AS 'com.esri.hadoop.hive.ST_SetSRID'")
cur.execute("CREATE TEMPORARY FUNCTION ST_GeodesicLengthWGS84 AS 'com.esri.hadoop.hive.ST_GeodesicLengthWGS84'")
cur.execute("CREATE TEMPORARY FUNCTION ST_LineString AS 'com.esri.hadoop.hive.ST_LineString'")
cur.execute("CREATE TEMPORARY FUNCTION ST_AsBinary AS 'com.esri.hadoop.hive.ST_AsBinary'")
cur.execute("CREATE TEMPORARY FUNCTION ST_PointFromWKB AS 'com.esri.hadoop.hive.ST_PointFromWKB'")
cur.execute("CREATE TEMPORARY FUNCTION ST_GeomFromWKB AS 'com.esri.hadoop.hive.ST_GeomFromWKB'")
cur.execute("CREATE TEMPORARY FUNCTION ST_Contains AS 'com.esri.hadoop.hive.ST_Contains'")

In [4]:
# Create the tables in the format we want
username = "groupab"
#cur.execute(f"""DROP TABLE {username}.sbb_stops_final_region""")
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {username}.sbb_stops_final_region
(
    stop_id STRING,
    stop_name STRING,
    stop_lat FLOAT,
    stop_lon FLOAT
)
STORED AS PARQUET
""")

#cur.execute(f"""DROP TABLE {username}.sbb_stop_times_final_region""")
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {username}.sbb_stop_times_final_region
(
    trip_id STRING,
    stop_id STRING,
    arrival_time STRING,
    departure_time STRING
)
STORED AS PARQUET
""")

#cur.execute(f"""DROP TABLE {username}.sbb_stop_to_stop_final_region""")
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {username}.sbb_stop_to_stop_final_region
(
    stop_id_a STRING,
    stop_id_b STRING,
    distance FLOAT
)
STORED AS PARQUET
""")

In [7]:
#cur.execute(f"""
#SELECT *
#FROM
#    groupab.sbb_stops_final_region limit 5
#""")
#cur.fetchall()

[('Parent8501166', 'Romanel-sur-Lausanne', 46.56247, 6.603174),
 ('8592232', 'Savigny, Palaz', 46.54147, 6.735083),
 ('8592229', 'Savigny, Clair-Matin', 46.550232, 6.7469854),
 ('8592227', 'St-Sulpice VD, Pâqueret', 46.51665, 6.570449),
 ('8592207', 'Renens VD, Broye', 46.535408, 6.5982065)]

### Fill the tables


In [6]:
# Select all the stops in the region of interest (for example Lausanne region if OBJECT_ID =1)
cur.execute(f"""
INSERT OVERWRITE TABLE {username}.sbb_stops_final_region
SELECT stop_id, stop_name, stop_lat, stop_lon
FROM
    {default_db}.sbb_orc_stops stops JOIN {default_db}.geo_shapes shapes
    WHERE ST_Contains(shapes.geometry, ST_Point(stops.stop_lon,stops.stop_lat)) 
    AND objectid={OBJECT_ID}
""")

In [8]:
# Select all the pairs of stop within walking (500 meters) distance of each other
cur.execute(f"""
INSERT OVERWRITE TABLE {username}.sbb_stop_to_stop_final_region
SELECT stop_id_a, stop_id_b, distance FROM
(
SELECT
    a.stop_id AS stop_id_a, b.stop_id AS stop_id_b,
    ST_GeodesicLengthWGS84(
        ST_SetSRID(ST_LineString(a.stop_lon, a.stop_lat, b.stop_lon, b.stop_lat), 4326)
    ) AS distance
FROM {username}.sbb_stops_final_region a JOIN {username}.sbb_stops_final_region b
WHERE
    a.stop_id != b.stop_id
) tmp WHERE distance < 500
""")

In [9]:
# Select the stop times
query=f"""
INSERT OVERWRITE TABLE {username}.sbb_stop_times_final_region
SELECT
    trip_id,
    stop_id,
    departure_time,
    arrival_time
FROM {default_db}.sbb_orc_stop_times
WHERE stop_id IN (SELECT DISTINCT stop_id FROM {username}.sbb_stops_final_region)
AND   trip_id IN (SELECT DISTINCT b.trip_id
        FROM {default_db}.sbb_orc_calendar a INNER JOIN {default_db}.sbb_orc_trips b
        WHERE a.monday='TRUE' AND b.service_id = a.service_id)
"""
cur.execute(query)

In [10]:
# To store in a local pd dataframe 


def rename_col(df):
    new_names = [col.split('.')[1] for col in df.columns ]
    df.columns = new_names

stops_df = pd.read_sql(f"SELECT * FROM {username}.sbb_stops_final_region ", conn, columns=['stop_id', 'stop_name', 'stop_lat', 'stop_lon'])
rename_col(stops_df)
stop_to_stop_df = pd.read_sql(f"SELECT * FROM {username}.sbb_stop_to_stop_final_region ", conn , columns =['stop_id_a', 'stop_id_b', 'distance'])
rename_col(stop_to_stop_df)
stop_times_df = pd.read_sql(f"SELECT * FROM {username}.sbb_stop_times_final_region ", conn, columns =['trip_idddd', 'stop_id','arrival_time','departure_time'])
rename_col(stop_times_df)

route_desc_df = pd.read_csv("data/route_desc.csv")
routes_df = pd.read_csv("data/routes.csv")